# Validate Migration

Compares source vs target: version counts, metrics, params, lineage tags, aliases. Runs a quick inference test on the target model.

## Configuration

Read widget parameters: source/target catalogs, schema, volume catalog, and import volume name.

In [ ]:
dbutils.widgets.text("source_catalog", "", "Source catalog")
dbutils.widgets.text("target_catalog", "", "Target catalog")
dbutils.widgets.text("source_schema", "default", "Source schema")
dbutils.widgets.text("target_schema", "default", "Target schema")
dbutils.widgets.text("import_volume", "model_imports", "Import volume name")
dbutils.widgets.text("volume_catalog", "", "Catalog holding import volume (empty = target_catalog)")

SOURCE_CATALOG = dbutils.widgets.get("source_catalog").strip()
TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
TARGET_SCHEMA = dbutils.widgets.get("target_schema").strip()
IMPORT_VOLUME = dbutils.widgets.get("import_volume").strip()
VOLUME_CATALOG = dbutils.widgets.get("volume_catalog").strip() or TARGET_CATALOG
VOLUME_ROOT = f"/Volumes/{VOLUME_CATALOG}/{TARGET_SCHEMA}/{IMPORT_VOLUME}"

if not TARGET_CATALOG:
    raise ValueError("Set target_catalog")
print(f"Volume: {VOLUME_ROOT}")

## MLflow client and validation helpers

Set up the MLflow UC registry client, define comparison functions for metrics and params, and load the import manifest.

In [ ]:
import mlflow, json, os, time, math
import numpy as np, pandas as pd
from mlflow import MlflowClient
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

def _retry(fn, retries=3, backoff_s=1.0):
    for i in range(retries):
        try: return fn()
        except Exception as e:
            if i == retries - 1: raise
            time.sleep(backoff_s * (2 ** i))

def safe_isclose(a, b, abs_tol=1e-5, rel_tol=1e-5):
    try: return math.isclose(float(a), float(b), abs_tol=abs_tol, rel_tol=rel_tol)
    except (TypeError, ValueError): return str(a).strip() == str(b).strip()

def compare_metrics(exp, rec):
    if not exp: return True
    return all(k in rec and safe_isclose(exp[k], rec[k]) for k in exp)

def compare_params(exp, rec):
    if not exp: return True
    return all(k in rec and (safe_isclose(exp[k], rec[k]) or str(exp[k]).strip() == str(rec[k]).strip()) for k in exp)

def list_aliases(name):
    """Return {alias_name: version_str} for a registered model; {} on lookup failure."""
    try: rm = client.get_registered_model(name)
    except Exception: return {}
    al = getattr(rm, "aliases", None) or []
    out = {}
    if isinstance(al, dict):
        for k, v in al.items(): out[str(k)] = str(v)
    else:
        for a in al:
            if hasattr(a, "alias"): out[a.alias] = str(a.version)
            elif isinstance(a, dict): out[a["alias"]] = str(a["version"])
    return out

with open(f"{VOLUME_ROOT}/manifest.json") as f:
    models_info = json.load(f)["models"]
print(f"Models to validate: {len(models_info)}")

## Validate each model

For each migrated model: compares version counts, metrics, params, and lineage tags between source metadata and target registry. Checks alias assignments and runs a quick inference test on the latest target version.

In [ ]:
for m_info in models_info:
    source_model = m_info["source"]
    target_short = m_info["target"].split(".")[-1] if m_info.get("target") else source_model.split(".")[-1]
    target_model = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{target_short}"
    model_root = f"{VOLUME_ROOT}/{m_info['dir']}"
    expected_versions = m_info["versions"]
    print(f"\n--- {source_model} vs {target_model} ---")
    tgt_all = _retry(lambda: list(client.search_model_versions(filter_string=f"name='{target_model}'")))
    tgt_sorted = sorted(tgt_all, key=lambda v: int(v.version))
    v_ok = len(tgt_sorted) == expected_versions
    print(f"  Version count: expected={expected_versions} target={len(tgt_sorted)}  {'PASS' if v_ok else 'FAIL'}")
    version_dirs = sorted([d for d in os.listdir(model_root) if d.startswith("v") and os.path.isdir(f"{model_root}/{d}")], key=lambda x: int(x[1:]))
    m_ok = p_ok = t_ok = 0
    for vd in version_dirs:
        with open(f"{model_root}/{vd}/metadata.json") as f: meta = json.load(f)
        src_rid, src_ver = meta.get("source_run_id"), meta.get("source_version")
        tv = None
        for t in tgt_sorted:
            try:
                tt = dict(client.get_run(t.run_id).data.tags)
                if tt.get("migration.source_run_id") == src_rid or tt.get("migration.source_version") == str(src_ver): tv = t; break
            except Exception: pass
        if tv:
            tr = client.get_run(tv.run_id)
            if compare_metrics(meta.get("metrics", {}), dict(tr.data.metrics)): m_ok += 1
            if compare_params(meta.get("params", {}), dict(tr.data.params)): p_ok += 1
            if "migration.source_run_id" in dict(tr.data.tags): t_ok += 1
    total = len(version_dirs)
    print(f"  Metrics:  {m_ok}/{total}  Params: {p_ok}/{total}  Lineage: {t_ok}/{total}")
    src_aliases = list_aliases(source_model)
    tgt_aliases = list_aliases(target_model)
    if not src_aliases and not tgt_aliases:
        print("  Aliases: none set on source or target")
    else:
        for a in sorted(src_aliases):
            sv, tv = src_aliases[a], tgt_aliases.get(a)
            if tv is None: print(f"  {a}: source=v{sv} target=—  MISSING on target")
            else: print(f"  {a}: v{tv}  OK")
        for a in sorted(set(tgt_aliases) - set(src_aliases)):
            print(f"  {a}: v{tgt_aliases[a]} on target only  WARN (not on source)")
    try:
        loaded = mlflow.sklearn.load_model(f"models:/{target_model}/{tgt_sorted[-1].version}")
        nf = getattr(loaded, "n_features_in_", 5)
        X = np.random.default_rng(42).normal(0, 1, (2, max(1, nf))).astype(np.float64)
        _ = loaded.predict(X)
        print(f"  Inference: PASS")
    except Exception as e: print(f"  Inference: FAIL — {e}")
print("\nValidation done.")